In [3]:
import os
import time
import random
import warnings
from datetime import date
import gc

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree # type: ignore <- pylance milně hlásí chybu
from pathlib import Path
import time
import sys
import polars as pl
import polars.selectors as cs
import json
import pickle
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patheffects as path_effects
from ydata_profiling import ProfileReport
import geopandas as gpd


current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
sys.path.append(str(current_dir.parent))
from utils import *
from schemas import *
from clean import *
from visualisation_utils import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')
# os.chdir(r'C:\Users\adamp\Projects\CVUT_BAP')
SEED=42
PRINT = False
DATA_PATH = r"E:\CVUT_BAP\kod\data\processed\mereni.parquet"
TEMP_PATH = r"E:\CVUT_BAP\kod\data\processed\mereni.tmp.parquet"

# Načtení dat

In [2]:
df = pl.read_parquet(r'kod\data\extracted\data_z_mericich_pristroju\parquet', schema=mereni_schema)
if PRINT: describe(df, True)

# Přetypování sloupců

In [3]:
df = cast_mereni(df)

# Vyber mereni, ktera maji protejsek v prohlidkach (zajisteni stejnych filtru)

In [4]:
lf_prohlidky = pl.scan_parquet(r'E:\CVUT_BAP\kod\data\processed\prohlidky.parquet')
id_prohlidky_series = lf_prohlidky.select('CisloProtokolu').unique().collect(engine='streaming').get_column('CisloProtokolu')

if PRINT:
    id_prohlidky = set(id_prohlidky_series)
    id_mereni = set(df.get_column('CisloProtokolu'))
    orphan = list(id_prohlidky - id_mereni)
    lf_prohlidky.filter(pl.col('CisloProtokolu').is_in(orphan)).collect(engine='streaming')['DatumProhlidky'].value_counts(sort=True)

df = df.join(id_prohlidky_series.to_frame("CisloProtokolu"), on="CisloProtokolu", how="semi")

# Odstraneni nekterych sloupcu
### Rok_vyroby

In [5]:
if PRINT: print(f'Podil mereni, ktera maji rok vyroby mensi nez 1000: {len(df.filter(pl.col('Vozidlo_RokVyroby') < 1000)) / df.height * 100:.2f} %')

### Vysledek_RidiciJednotka

In [6]:
if PRINT: 
    print(df.select('Vysledek_RidiciJednotkaStav', 'Vysledek_RidiciJednotka').filter(pl.col('Vysledek_RidiciJednotkaStav') == 1)['Vysledek_RidiciJednotka'].value_counts(sort=True)['Vysledek_RidiciJednotka'].to_list()[:10])
    print(df.select('Vysledek_RidiciJednotkaStav', 'Vysledek_RidiciJednotka').filter(pl.col('Vysledek_RidiciJednotkaStav') == 2)['Vysledek_RidiciJednotka'].value_counts(sort=True)['Vysledek_RidiciJednotka'].to_list()[:10])
    print(df.select('Vysledek_RidiciJednotkaStav', 'Vysledek_RidiciJednotka').filter(pl.col('Vysledek_RidiciJednotkaStav') == 3)['Vysledek_RidiciJednotka'].value_counts(sort=True)['Vysledek_RidiciJednotka'].to_list()[:10])

### Obd_VypisDTC

In [7]:
if PRINT: 
    obd_vypis_dtc_non_null = df['Obd_VypisDtc'].filter(df['Obd_VypisDtc'].is_not_null())
    print(f'Podil, kdy obd vypis dtc neni null: {len(obd_vypis_dtc_non_null) / df.height * 100:.2f} %')
    print(obd_vypis_dtc_non_null.to_list()[:10])

### J1939
- Odstraneni radku, ktere hodnoty obsahuji

In [8]:
j1939_df = df.filter(
    pl.any_horizontal(
        pl.col("^.*J1939.*$").is_not_null()
    )
)
if PRINT: short_display(j1939_df)
# Modifikace df
df = df.join(j1939_df, on="CisloProtokolu", how="anti")

### Plyn_PocetVyusteni

In [9]:
if PRINT: df['Plyn_PocetVyusteni'].value_counts()

### PristiProhlidka

In [10]:
if PRINT: df.filter(pl.col('PristiProhlidka').is_not_null())['Vysledek_Vyhovuje'].value_counts()

In [11]:
duplicate_cols = ['DatumProhlidky', 'Zahajeni', 'Ukonceni', 'OdpovednaOsoba', 'Prohlidka_DatumProhlidky', 'Vozidlo_Vin', 'Vozidlo_Znacka', 'Vozidlo_ObchodniOznaceni', 'Vozidlo_TypMotoru', 'Vozidlo_CisloMotoru', 'Vozidlo_Odometer', 'Vozidlo_DatumPrvniRegistrace', 'Vozidlo_Palivo', 'Obd_Vin']
irrelevant_cols = ['Prohlidka_CisloProtokolu', 'Vozidlo_RokVyroby', 'Vysledek_RidiciJednotka', 'Vysledek_TesnostPlynovehoZarizeni', 'PristiProhlidka', 'Benzin_Palivo', 'Nafta_Palivo']
j1939_cols = ['Obd_Readiness_J1939_AC_Podporovano', 'Obd_Readiness_J1939_AC_Otestovano', 'Obd_Readiness_J1939_BOOST_Podporovano', 'Obd_Readiness_J1939_BOOST_Otestovano', 'Obd_Readiness_J1939_CAT-FUNC_Podporovano', 'Obd_Readiness_J1939_CAT-FUNC_Otestovano', 'Obd_Readiness_J1939_COLD_Podporovano', 'Obd_Readiness_J1939_COLD_Otestovano', 'Obd_Readiness_J1939_COMP_Podporovano', 'Obd_Readiness_J1939_COMP_Otestovano', 'Obd_Readiness_J1939_DPF_Podporovano', 'Obd_Readiness_J1939_DPF_Otestovano', 'Obd_Readiness_J1939_EGR-VVT_Podporovano', 'Obd_Readiness_J1939_EGR-VVT_Otestovano', 'Obd_Readiness_J1939_EGS-FUNC_Podporovano', 'Obd_Readiness_J1939_EGS-FUNC_Otestovano', 'Obd_Readiness_J1939_EGS-HEAT_Podporovano', 'Obd_Readiness_J1939_EGS-HEAT_Otestovano', 'Obd_Readiness_J1939_EVAP_Podporovano', 'Obd_Readiness_J1939_EVAP_Otestovano', 'Obd_Readiness_J1939_FUEL_Podporovano', 'Obd_Readiness_J1939_FUEL_Otestovano', 'Obd_Readiness_J1939_HCAT_Podporovano', 'Obd_Readiness_J1939_HCAT_Otestovano', 'Obd_Readiness_J1939_MISF_Podporovano', 'Obd_Readiness_J1939_MISF_Otestovano', 'Obd_Readiness_J1939_NM-HC_Podporovano', 'Obd_Readiness_J1939_NM-HC_Otestovano', 'Obd_Readiness_J1939_NOX_Podporovano', 'Obd_Readiness_J1939_NOX_Otestovano', 'Obd_Readiness_J1939_SAS_Podporovano', 'Obd_Readiness_J1939_SAS_Otestovano']
plyn_cols = ['Plyn_Palivo', 'Plyn_PocetVyusteni', 'Plyn_OtackyVolnobezne_CO_Max_Hodnota', 'Plyn_OtackyVolnobezne_CO_Max_RucniZadani', 'Plyn_OtackyVolnobezne_CO_Hodnota', 'Plyn_OtackyVolnobezne_CO_Vysledek', 'Plyn_OtackyVolnobezne_CO2_Hodnota', 'Plyn_OtackyVolnobezne_CO2_Vysledek', 'Plyn_OtackyVolnobezne_COCOOR_Hodnota', 'Plyn_OtackyVolnobezne_COCOOR_Vysledek', 'Plyn_OtackyVolnobezne_HC_Max_Hodnota', 'Plyn_OtackyVolnobezne_HC_Max_RucniZadani', 'Plyn_OtackyVolnobezne_HC_Hodnota', 'Plyn_OtackyVolnobezne_HC_Vysledek', 'Plyn_OtackyVolnobezne_LAMBDA_Min_Hodnota', 'Plyn_OtackyVolnobezne_LAMBDA_Min_RucniZadani', 'Plyn_OtackyVolnobezne_LAMBDA_Max_Hodnota', 'Plyn_OtackyVolnobezne_LAMBDA_Max_RucniZadani', 'Plyn_OtackyVolnobezne_LAMBDA_Hodnota', 'Plyn_OtackyVolnobezne_LAMBDA_Vysledek', 'Plyn_OtackyVolnobezne_N_Min_Hodnota', 'Plyn_OtackyVolnobezne_N_Min_RucniZadani', 'Plyn_OtackyVolnobezne_N_Max_Hodnota', 'Plyn_OtackyVolnobezne_N_Max_RucniZadani', 'Plyn_OtackyVolnobezne_N_Hodnota', 'Plyn_OtackyVolnobezne_N_Vysledek', 'Plyn_OtackyVolnobezne_NOX_Hodnota', 'Plyn_OtackyVolnobezne_NOX_Vysledek', 'Plyn_OtackyVolnobezne_O2_Hodnota', 'Plyn_OtackyVolnobezne_O2_Vysledek', 'Plyn_OtackyVolnobezne_TPS_Hodnota', 'Plyn_OtackyVolnobezne_TPS_Vysledek', 'Plyn_OtackyZvysene_CO_Max_Hodnota', 'Plyn_OtackyZvysene_CO_Max_RucniZadani', 'Plyn_OtackyZvysene_CO_Hodnota', 'Plyn_OtackyZvysene_CO_Vysledek', 'Plyn_OtackyZvysene_CO2_Hodnota', 'Plyn_OtackyZvysene_CO2_Vysledek', 'Plyn_OtackyZvysene_COCOOR_Hodnota', 'Plyn_OtackyZvysene_COCOOR_Vysledek', 'Plyn_OtackyZvysene_HC_Max_Hodnota', 'Plyn_OtackyZvysene_HC_Max_RucniZadani', 'Plyn_OtackyZvysene_HC_Hodnota', 'Plyn_OtackyZvysene_HC_Vysledek', 'Plyn_OtackyZvysene_LAMBDA_Min_Hodnota', 'Plyn_OtackyZvysene_LAMBDA_Min_RucniZadani', 'Plyn_OtackyZvysene_LAMBDA_Max_Hodnota', 'Plyn_OtackyZvysene_LAMBDA_Max_RucniZadani', 'Plyn_OtackyZvysene_LAMBDA_Hodnota', 'Plyn_OtackyZvysene_LAMBDA_Vysledek', 'Plyn_OtackyZvysene_N_Min_Hodnota', 'Plyn_OtackyZvysene_N_Min_RucniZadani', 'Plyn_OtackyZvysene_N_Max_Hodnota', 'Plyn_OtackyZvysene_N_Max_RucniZadani', 'Plyn_OtackyZvysene_N_Hodnota', 'Plyn_OtackyZvysene_N_Vysledek', 'Plyn_OtackyZvysene_NOX_Hodnota', 'Plyn_OtackyZvysene_NOX_Vysledek', 'Plyn_OtackyZvysene_O2_Hodnota', 'Plyn_OtackyZvysene_O2_Vysledek', 'Plyn_OtackyZvysene_TPS_Hodnota', 'Plyn_OtackyZvysene_TPS_Vysledek', 'Plyn_Nadrz_Vyrobce', 'Plyn_Nadrz_Homologace', 'Plyn_Nadrz_Zivotnost', 'Plyn_Nadrz_Kontrola']
df = df.drop(duplicate_cols + irrelevant_cols + j1939_cols + plyn_cols)

# Unikatnost cisla protokolu
- nasledne odstraneno stanice cislo

In [13]:
if PRINT: print(f'Délka datasetu před odstraněním duplicitních řádků: {df.height}')

df = df.unique()
if PRINT: print(f'Délka datasetu po odstranění duplicitních řádků: {df.height}')

non_unique = df.select(pl.col('CisloProtokolu')).filter(pl.col('CisloProtokolu').is_duplicated())['CisloProtokolu'].unique().to_list()
if PRINT: short_display(df.filter(pl.col('CisloProtokolu').is_in(non_unique)).sort(by=['CisloProtokolu', 'StaniceCislo']))

df = df.filter(~pl.col('CisloProtokolu').is_in(non_unique) | (pl.col('CisloProtokolu').str.split('-').list.get(1).cast(pl.Int32) == pl.col("StaniceCislo")))
if PRINT: print(f'Délka datasetu po odstranění vsech duplicitnich cisel protokolu: {df.height}')
df = df.drop('StaniceCislo')

# Serazeni sloupcu podle textu prace

In [14]:
df = df.select([
    # Identifikace měření
    "CisloProtokolu",

    # Měřicí přístroj
    "MericiPristroj_Vyrobce",
    "MericiPristroj_Typ",
    "MericiPristroj_Verze",
    "MericiPristroj_OBD",
    "MericiPristroj_VerzeSoftware",

    # Výsledek měření
    "Vysledek_VisualniKontrola",
    "Vysledek_Readiness",
    "Vysledek_RidiciJednotkaStav",
    "Vysledek_Mil",
    "Vysledek_Vyhovuje",
    "Obd_Readiness_Vysledek",
    "Zavady",
    "Poznamky",

    # Emisní systém (rizenyOBD)
    "EmisniSystem",
    "Obd_KomunikacniProtokol",
    "Obd_KontrolaMil",
    "Obd_PocetDtc",
    "Obd_VypisDtc",
    "Obd_VzdalenostDtc",
    "Obd_CasDtc",
    *sorted([c for c in df.columns if "Obd_Readiness_Zazeh" in c]),
    *sorted([c for c in df.columns if "Obd_Readiness_Vznet" in c]),

    # Detail měření dle typu paliva - Benzin
    "Benzin_PocetVyusteni",
    *sorted([c for c in df.columns if c.startswith("Benzin_OtackyVolnobezne")]),
    *sorted([c for c in df.columns if c.startswith("Benzin_OtackyZvysene")]),

    # Detail měření dle typu paliva - Nafta
    "Nafta_PocetVyusteni",
    *sorted([c for c in df.columns if c.startswith("Nafta_MereniVznetLimit")]),
    *sorted([c for c in df.columns if c.startswith("Nafta_MereniPrumer")]),
    *sorted([c for c in df.columns if c.startswith("Nafta_Mereni0")]),
    *sorted([c for c in df.columns if c.startswith("Nafta_Mereni1")]),
    *sorted([c for c in df.columns if c.startswith("Nafta_Mereni2")]),
    *sorted([c for c in df.columns if c.startswith("Nafta_Mereni3")])
])

In [15]:
if PRINT: schema_description(df)
df.write_parquet(r"E:\CVUT_BAP\kod\data\processed\mereni_tmp.parquet")

# Chybejici hodnoty

In [16]:
if PRINT: 
    # Definice agregačních skupin
    aggregation_groups = {
        'Obd_Readiness_Zazeh': [c for c in df.columns if 'Obd_Readiness_Zazeh' in c],
        'Obd_Readiness_Vznet': [c for c in df.columns if 'Obd_Readiness_Vznet' in c],
        'Benzin_OtackyVolnobezne': [c for c in df.columns if c.startswith('Benzin_OtackyVolnobezne_')],
        'Benzin_OtackyZvysene': [c for c in df.columns if c.startswith('Benzin_OtackyZvysene_')],
        'Nafta_MereniVznetLimit': [c for c in df.columns if c.startswith('Nafta_MereniVznetLimit_')],
        'Nafta_MereniPrumer': [c for c in df.columns if c.startswith('Nafta_MereniPrumer_')],
        'Nafta_Mereni0': [c for c in df.columns if c.startswith('Nafta_Mereni0_')],
        'Nafta_Mereni1': [c for c in df.columns if c.startswith('Nafta_Mereni1_')],
        'Nafta_Mereni2': [c for c in df.columns if c.startswith('Nafta_Mereni2_')],
        'Nafta_Mereni3': [c for c in df.columns if c.startswith('Nafta_Mereni3_')],
    }

    # Inverzní mapování: sloupec -> název skupiny
    col_to_group = {col: g_name for g_name, g_cols in aggregation_groups.items() for col in g_cols}

    # Výpočet metrik se zachováním pořadí z df.columns
    total_rows = len(df)
    final_labels = []
    final_ratios = []
    processed_groups = set()

    for col in df.columns:
        if col in col_to_group:
            group_name = col_to_group[col]
            if group_name not in processed_groups:
                # Výpočet pro celou skupinu (aspoň jedna ne-null hodnota)
                group_cols = aggregation_groups[group_name]
                any_present_count = df.select(
                    pl.any_horizontal(pl.col(group_cols).is_not_null())
                ).sum().item()
                final_labels.append(group_name)
                final_ratios.append(any_present_count / total_rows if total_rows > 0 else 0)
                processed_groups.add(group_name)
        else:
            # Samostatný sloupec
            non_null_count = total_rows - df[col].null_count()
            final_labels.append(col)
            final_ratios.append(non_null_count / total_rows if total_rows > 0 else 0)

    # Kategorizace (6 skupin včetně Vozidla)
    group_descriptions = [
        'Identifikace měření',
        'Měřicí přístroj',
        'Výsledek',  
        'Emisní systém', 
        'Detail měření'  
    ]

    def map_to_main_category(label):
        if label == 'CisloProtokolu': return 0
        if label.startswith('MericiPristroj'): return 1
        if any(x in label for x in ['Vysledek', 'PristiProhlidka', 'Zavady', 'Poznamky']): return 2
        if label.startswith('EmisniSystem') or label.startswith('Obd'): return 3
        if label.startswith('Benzin') or label.startswith('Nafta'): return 4
        return 3

    group_indices = [map_to_main_category(l) for l in final_labels]

    # Generování grafu
    horizontal_bar(
        labels=final_labels, 
        counts=final_ratios, 
        title='Poměr přítomných údajů v měřeních', 
        save_path='kod/explorace/mereni_grafy/mereni_pritomnost.svg', 
        decimals=3, 
        height=18, 
        group_indices=group_indices, 
        group_descriptions=group_descriptions
    )

### Emisni systemy

In [17]:
if PRINT: df['EmisniSystem'].value_counts(sort=True).with_columns(pl.col('count') / df.height)

### Pocet mereni u nafty

In [18]:
if PRINT: 
    print(f'Pomer dieselu, co ma alespon 2 mereni: {0.264 / 0.456 * 100:.2f} %')
    print(f'Pomer dieselu, co ma 3 a 4 mereni: {0.146 / 0.456 * 100:.2f} %')

# Ulozeni dat

In [ ]:
df.write_parquet(DATA_PATH)

In [126]:
df = pl.read_parquet(DATA_PATH)

# Odstraneni chybnych zaznamu
## Zaporne hodnoty

In [127]:
if PRINT: print(f'Pocet zaznamu se zapornou hodnou {df.filter(pl.any_horizontal(cs.numeric() < 0)).height}')
df = df.filter(pl.any_horizontal(~(cs.numeric() < 0)))

Pocet zaznamu se zapornou hodnou 7351


## Filtrace zaznamu predstavujicich konkretni mereni pro benzin nebo naftu

In [ ]:
required_benzin = [
    # Benzin Vždy (Volnoběžné)
    "Benzin_OtackyVolnobezne_CO_Hodnota",
    "Benzin_OtackyVolnobezne_CO_Max_Hodnota",
    "Benzin_OtackyVolnobezne_N_Hodnota",
    "Benzin_OtackyVolnobezne_N_Min_Hodnota",
    "Benzin_OtackyVolnobezne_N_Max_Hodnota",
    
    # Benzin Zvýšené
    "Benzin_OtackyZvysene_LAMBDA_Hodnota",
    "Benzin_OtackyZvysene_LAMBDA_Min_Hodnota",
    "Benzin_OtackyZvysene_LAMBDA_Max_Hodnota",
    "Benzin_OtackyZvysene_CO_Max_Hodnota",
    "Benzin_OtackyZvysene_N_Hodnota",
    "Benzin_OtackyZvysene_N_Min_Hodnota",
    "Benzin_OtackyZvysene_N_Max_Hodnota"
]

required_nafta = [
    # Nafta Průměr
    "Nafta_MereniPrumer_CasAkcelerace_Hodnota",
    "Nafta_MereniPrumer_Kourivost_Hodnota",
    "Nafta_MereniPrumer_OtackyVolnobezne_Hodnota",
    "Nafta_MereniPrumer_OtackyPrebehove_Hodnota",
    
    # Nafta Limit Max
    "Nafta_MereniVznetLimit_CasAkcelerace_Max_Hodnota",
    "Nafta_MereniVznetLimit_Kourivost_Max_Hodnota",
    "Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota",
    "Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota",
    
    # Nafta Limit Min
    "Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota",
    "Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota"
]

benzin_mask = pl.all_horizontal(pl.col(required_benzin).is_not_null())
nafta_mask = pl.all_horizontal(pl.col(required_nafta).is_not_null())
if PRINT:
    print(f'Pocet neuplnych zaznamu: {df.filter(~(benzin_mask | nafta_mask)).height}')
    print(f'Pocet zaznamu s benzinem i naftou: {df.filter((pl.col('Nafta_PocetVyusteni') > 0) & (pl.col('Benzin_PocetVyusteni') > 0)).height}')
df = df.filter(benzin_mask | nafta_mask)

Pocet neuplnych zaznamu: 86801
Pocet zaznamu s benzinem i naftou: 0


## Kontrola limitu
### Benzin

In [ ]:
limits_bounds_benzin = [
    # (Název atributu limitu, Očekávané MIN limitu, Očekávané MAX limitu)
    ('Benzin_OtackyVolnobezne_CO_Max_Hodnota', 0.05, 5.0),
    ('Benzin_OtackyVolnobezne_N_Min_Hodnota', 300, 3000),
    ('Benzin_OtackyVolnobezne_N_Max_Hodnota', 300, 3000),
    ('Benzin_OtackyZvysene_CO_Max_Hodnota', 0.1, 1.0),
    ('Benzin_OtackyZvysene_LAMBDA_Min_Hodnota', 0.9, 1.00),
    ('Benzin_OtackyZvysene_LAMBDA_Max_Hodnota', 1.00, 1.1),
    ('Benzin_OtackyZvysene_N_Min_Hodnota', 1000, 10000),
    ('Benzin_OtackyZvysene_N_Max_Hodnota', 1000, 10000)
]

for limit_bound in limits_bounds_benzin:
    if PRINT: print(f'Pocet hodnot mimo limit pro {limit_bound[0]}: {df.filter(~pl.col(limit_bound[0]).is_between(limit_bound[1], limit_bound[2])).height}')
    df = df.filter(pl.col(limit_bound[0]).is_between(limit_bound[1], limit_bound[2]) | nafta_mask)

Pocet hodnot mimo limit pro Benzin_OtackyVolnobezne_CO_Max_Hodnota: 17535
Pocet hodnot mimo limit pro Benzin_OtackyVolnobezne_N_Min_Hodnota: 529
Pocet hodnot mimo limit pro Benzin_OtackyVolnobezne_N_Max_Hodnota: 384
Pocet hodnot mimo limit pro Benzin_OtackyZvysene_CO_Max_Hodnota: 90333
Pocet hodnot mimo limit pro Benzin_OtackyZvysene_LAMBDA_Min_Hodnota: 17252
Pocet hodnot mimo limit pro Benzin_OtackyZvysene_LAMBDA_Max_Hodnota: 2847
Pocet hodnot mimo limit pro Benzin_OtackyZvysene_N_Min_Hodnota: 273
Pocet hodnot mimo limit pro Benzin_OtackyZvysene_N_Max_Hodnota: 552


### Nafta

In [ ]:
limits_bounds_nafta = [
    # (Název atributu limitu, Očekávané MIN limitu, Očekávané MAX limitu)
    ('Nafta_MereniVznetLimit_CasAkcelerace_Max_Hodnota', 0.1, 10.0),
    ('Nafta_MereniVznetLimit_Kourivost_Max_Hodnota', 0.1, 3.0),
    ('Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota', 300, 3000),
    ('Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota', 300, 3000),
    ('Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota', 1000, 10000),
    ('Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota', 1000, 10000)
]

for limit_bound in limits_bounds_nafta:
    if PRINT: print(f'Pocet hodnot mimo limit pro {limit_bound[0]}: {df.filter(~pl.col(limit_bound[0]).is_between(limit_bound[1], limit_bound[2])).height}')
    df = df.filter(pl.col(limit_bound[0]).is_between(limit_bound[1], limit_bound[2]) | benzin_mask)

Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_CasAkcelerace_Max_Hodnota: 441
Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_Kourivost_Max_Hodnota: 171065
Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota: 153
Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota: 79
Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota: 80
Pocet hodnot mimo limit pro Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota: 131


### Rovnosti

In [ ]:
# Seznam dvojic limitů pro kontrolu integrity (MIN == MAX je chyba)
check_limits_integrity = [
    ("Benzin_OtackyVolnobezne_N_Min_Hodnota", "Benzin_OtackyVolnobezne_N_Max_Hodnota"),
    ("Benzin_OtackyZvysene_LAMBDA_Min_Hodnota", "Benzin_OtackyZvysene_LAMBDA_Max_Hodnota"),
    ("Benzin_OtackyZvysene_N_Min_Hodnota", "Benzin_OtackyZvysene_N_Max_Hodnota"),
    ("Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota", "Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota"),
    ("Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota", "Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota")
]

for min_col, max_col in check_limits_integrity:
    if PRINT: print(f'Pocet hodnot s nulovou delkou intervalu {min_col}, {max_col}: {df.filter(pl.col(min_col) == pl.col(max_col)).height}')
    df = df.filter((pl.col(min_col).ne(pl.col(max_col))).fill_null(True))

Pocet hodnot s nulovou delkou intervalu Benzin_OtackyVolnobezne_N_Min_Hodnota, Benzin_OtackyVolnobezne_N_Max_Hodnota: 15
Pocet hodnot s nulovou delkou intervalu Benzin_OtackyZvysene_LAMBDA_Min_Hodnota, Benzin_OtackyZvysene_LAMBDA_Max_Hodnota: 4
Pocet hodnot s nulovou delkou intervalu Benzin_OtackyZvysene_N_Min_Hodnota, Benzin_OtackyZvysene_N_Max_Hodnota: 193
Pocet hodnot s nulovou delkou intervalu Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota, Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota: 11
Pocet hodnot s nulovou delkou intervalu Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota, Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota: 17


## Pro dalsi analyzu uvazovany pouze Rizene emisni systemy
- odstraneny i zaznamy, ktere maji skoro same null (pro analyzu vsak mene relevantni - pouze jako vysledek)
- diky tomu neni mereno HC

In [136]:
if PRINT: print(f'Podil rizenych: {df['EmisniSystem'].value_counts().filter(pl.col('EmisniSystem').cast(pl.String).str.contains('Rizeny'))['count'].sum() / df.height * 100:.2f} %')
df = df.filter(pl.col('EmisniSystem').cast(pl.String).str.contains('Rizeny'))

Podil rizenych: 100.00 %


# Extrakce dulezitych informaci

## Normalizace hodnot

In [139]:
# Definice mapování pro normalizaci
all_mappings = [
    ("Benzin_OtackyVolnobezne_CO_Hodnota", None, "Benzin_OtackyVolnobezne_CO_Max_Hodnota"),
    ("Benzin_OtackyVolnobezne_N_Hodnota", "Benzin_OtackyVolnobezne_N_Min_Hodnota", "Benzin_OtackyVolnobezne_N_Max_Hodnota"),
    ("Benzin_OtackyZvysene_LAMBDA_Hodnota", "Benzin_OtackyZvysene_LAMBDA_Min_Hodnota", "Benzin_OtackyZvysene_LAMBDA_Max_Hodnota"),
    ("Benzin_OtackyZvysene_CO_Hodnota", None, "Benzin_OtackyZvysene_CO_Max_Hodnota"),
    ("Benzin_OtackyZvysene_N_Hodnota", "Benzin_OtackyZvysene_N_Min_Hodnota", "Benzin_OtackyZvysene_N_Max_Hodnota"),
    ("Nafta_MereniPrumer_CasAkcelerace_Hodnota", None, "Nafta_MereniVznetLimit_CasAkcelerace_Max_Hodnota"),
    ("Nafta_MereniPrumer_Kourivost_Hodnota", None, "Nafta_MereniVznetLimit_Kourivost_Max_Hodnota"),
    ("Nafta_MereniPrumer_OtackyVolnobezne_Hodnota", "Nafta_MereniVznetLimit_OtackyVolnobezne_Min_Hodnota", "Nafta_MereniVznetLimit_OtackyVolnobezne_Max_Hodnota"),
    ("Nafta_MereniPrumer_OtackyPrebehove_Hodnota", "Nafta_MereniVznetLimit_OtackyPrebehove_Min_Hodnota", "Nafta_MereniVznetLimit_OtackyPrebehove_Max_Hodnota")
]

exprs = []
for val_col, min_col, max_col in all_mappings:
    low = pl.col(min_col) if min_col else pl.lit(0)
    high = pl.col(max_col)
    val = pl.col(val_col)
    
    # Normalizace
    norm_expr = ((val - low) / (high - low)).alias(f"{val_col}_Norm")
    exprs.append(norm_expr)

df = df.with_columns(exprs).drop((cs.starts_with('Benzin_') | cs.starts_with('Nafta_')) & (~cs.ends_with('PocetVyusteni')) & (~cs.ends_with('_Norm')))

## Spojeni informaci o mericim pristroji do Vyrobce a Typu

In [140]:
df = df.with_columns((pl.col('MericiPristroj_Vyrobce') + pl.col('MericiPristroj_Typ')).alias('MericiPristroj_Info')).drop(cs.contains("MericiPristroj") & (~cs.by_name("MericiPristroj_Info")))

## Pro nizkou ruznorodost ponechano pouze Obd_PocetDtc z DTC

In [141]:
df = df.drop(cs.contains('Dtc') & (~cs.by_name('Obd_PocetDtc')))

## Readiness podpory a otestovani prevedeno do poctu True

In [142]:
df = df.with_columns(pl.sum_horizontal(cs.ends_with("_Podporovano")).alias("Pocet_Podporovano"), pl.sum_horizontal(cs.ends_with("_Otestovano")).alias("Pocet_Otestovano"))
df = df.drop((cs.ends_with("_Podporovano") | cs.ends_with("_Otestovano")) & (~cs.by_name("Pocet_Podporovano")) & (~cs.by_name("Pocet_Otestovano")))

# Analyza benzinu

In [150]:
df_benzin = df.filter(pl.col('Benzin_PocetVyusteni') > 0).drop(cs.starts_with('Nafta'))

In [151]:
df_benzin.filter(pl.col('Vysledek_Vyhovuje') == True).select(cs.starts_with('Benzin')).describe()

statistic,Benzin_PocetVyusteni,Benzin_OtackyVolnobezne_CO_Hodnota_Norm,Benzin_OtackyVolnobezne_N_Hodnota_Norm,Benzin_OtackyZvysene_LAMBDA_Hodnota_Norm,Benzin_OtackyZvysene_CO_Hodnota_Norm,Benzin_OtackyZvysene_N_Hodnota_Norm
str,f64,f64,f64,f64,f64,f64
"""count""",8.454012e6,8.454012e6,8.454012e6,8.454012e6,8.454012e6,8.454012e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",1.00309,0.129253,0.513828,0.605711,0.326995,0.465718
"""std""",0.055525,0.521483,0.228209,11.609933,0.40097,0.30136
"""min""",1.0,-0.54,-15.5,-33.000031,-0.2,-40.0
"""25%""",1.0,0.0,0.4,0.483334,0.05,0.255
"""50%""",1.0,0.036667,0.5,0.55,0.2,0.455
"""75%""",1.0,0.16,0.608,0.683334,0.566667,0.665
"""max""",4.0,920.0,144.5,17050.517578,633.333313,100.900002
